# OCR + TTS Baseline

A simple English label reader for the MediConfirm comparison. The tool extracts printed text with Tesseract and uses OCR word coordinates and fixed label-layout rules to extract and read the medication name and strength with gTTS. It does not use an API key, match a medication list, explain a drug, or decide whether a box is correct.

Use the same reference list and equivalent tasks in both conditions. The evaluator supplies the list. No real medication is taken.


## 1. Install Python packages
Run this cell in the Anaconda/Jupyter environment used for this notebook. If widget packages were newly installed, restart Jupyter and the kernel before continuing.


In [ ]:
%pip install "anywidget>=0.9,<1" "ipywidgets>=8,<9" "Pillow>=10,<13" "gTTS>=2.5,<3" "pytesseract>=0.3.13,<1"


## 2. Install/check the OCR engine (Windows, macOS and Linux)

Tesseract is a separate program, not a built-in Python module. The next cell searches the active Anaconda environment, PATH and common operating-system locations. If it is missing in a Conda environment, it installs the same conda-forge package into the **current kernel environment**. This can take several minutes and needs internet. It does not require Homebrew.

For a clean, separate environment instead, use the included `environment.yml`: `conda env create -f environment.yml`, then `conda activate mediconfirm-baseline` and `jupyter notebook`.

Without Conda, use your operating system's Tesseract installation method: the Windows installer linked by the official Tesseract docs, Homebrew on macOS, or your Linux package manager. A custom executable path can be supplied below.


In [ ]:
import os
import sys
import shutil
import subprocess
import importlib
from pathlib import Path

candidates = [Path.cwd(), Path.cwd() / "baselines" / "ocr_tts", Path.cwd() / "ocr_tts"]
BASELINE_DIR = next((p for p in candidates if (p / "ocr_setup.py").is_file()), None)
if BASELINE_DIR is None:
    raise FileNotFoundError("Open this notebook from baselines/ocr_tts or from the repository root. Keep the helper files with it.")
if str(BASELINE_DIR) not in sys.path:
    sys.path.insert(0, str(BASELINE_DIR))
import ocr_setup
importlib.reload(ocr_setup)

TESSERACT_PATH = ""  # Optional custom executable, e.g. r"C:\Program Files\Tesseract-OCR\tesseract.exe"
INSTALL_MISSING_ENGINE = True
engine = ocr_setup.locate_engine(TESSERACT_PATH)
if engine is None and INSTALL_MISSING_ENGINE:
    conda = os.environ.get("CONDA_EXE") or shutil.which("conda")
    if conda and (Path(sys.prefix) / "conda-meta").is_dir():
        print("Installing Tesseract into the current kernel environment:", sys.prefix, flush=True)
        subprocess.run([conda, "install", "--prefix", sys.prefix, "-c", "conda-forge", "tesseract", "-y"], check=True)
    else:
        print("Conda installation is unavailable. Follow the non-Conda instructions above.")
version, engine = ocr_setup.configure_engine(TESSERACT_PATH)
print("Operating system:", sys.platform)
print("Python environment:", sys.prefix)
print("Tesseract:", version)
print("Executable:", engine)
print("English OCR is ready.")


## 3. Open the label reader

Allow camera access. Take a photo or upload one, listen to the extracted name and strength, and compare it with the reference list yourself. Retake as needed. Once you have selected your box, click **I selected this box — finish task** to record the simulated confirmation. The evaluator scores whether the selection was correct.

The camera stops after capture, cancellation, closing or leaving the page. A new audio response stops the previous one. If this cell is rerun, the previous widget is closed first.


In [ ]:
import sys
import importlib
from IPython.display import display

candidates = [Path.cwd(), Path.cwd() / "baselines" / "ocr_tts", Path.cwd() / "ocr_tts"]
BASELINE_DIR = next((p for p in candidates if (p / "baseline.py").is_file()), None)
if BASELINE_DIR is None:
    raise FileNotFoundError("Open this notebook from baselines/ocr_tts or from the repository root.")
if str(BASELINE_DIR) not in sys.path:
    sys.path.insert(0, str(BASELINE_DIR))
if "reader" in globals():
    reader.close()
import baseline
importlib.reload(baseline)
reader = baseline.BaselineReader(BASELINE_DIR / "baseline_records")
display(reader)


Full OCR text is kept under **View all extracted label text** for manual review. Only the extracted medication name and strength are narrated. If either cannot be extracted, the reader asks you to review the text or retake; it does not read the whole label or fill fields from the reference list.

## 4. Evaluation records

`baseline_records/tasks.json` retains attempts and click details. `baseline_records/summary.csv` contains one row per task. Timing starts at the first submitted photo and ends at the participant's explicit selection confirmation. It includes OCR/TTS wait, listening and retakes. Initial camera opening and box searching before first submission are outside this interval, matching the current MediConfirm website metric.

Click count is an interaction count, not a click rate. Audio-play events include automatic playback and are named separately in the raw log; report button click count separately. A completed task is not automatically a correct task. Record selection correctness independently.

OCR may return correct text, partial text, incorrect text or no text. Do not assume OCR always fails on blurred photos. Speech generation and camera permissions require a real-machine check. Logs contain no photos, audio, names or OCR text. Use one active widget per records folder.


In [ ]:
import csv
records_path = BASELINE_DIR / "baseline_records" / "summary.csv"
if records_path.exists():
    with records_path.open(newline="") as source:
        for row in csv.DictReader(source):
            print(row)
    print("CSV file:", records_path)
else:
    print("No baseline task records yet.")
